# Notebook 21 — Uncertainty in PSF outputs

Every MSV output today (`maturation_score`, `stress_score`, `drift_score`, etc.)
returns a single scalar. A researcher reading `stress_score = 0.72` cannot
distinguish "confidently in a stress state" from "model is uncertain and
true value could be anywhere from 0.4 to 0.9."

This notebook demonstrates PSF's v0.6 uncertainty layer:

1. **`ConformalPathwayPredictor`** — distribution-free prediction intervals
2. **`BootstrapMSV`** — non-parametric bootstrap over cells
3. **`BayesianPathwayGMM`** — posterior samples over subtype assignments
4. **`CalibrationReport`** — reliability diagrams, ECE, Brier score

Research use only. Not for clinical decision-making.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from pathway_subtyping.uncertainty import (
    BayesianPathwayGMM,
    BootstrapMSV,
    CalibrationReport,
    ConformalPathwayPredictor,
)

rng = np.random.default_rng(42)

## 1. Conformal prediction intervals

Wrap any scoring function with a split-conformal predictor and a held-out
calibration set. The returned intervals achieve the requested marginal
coverage (e.g., 90%) without any distributional assumption beyond
exchangeability.

In [ ]:
# Synthetic regression: pathway score depends linearly on a latent covariate
n = 3000
x = rng.uniform(-2, 2, size=(n, 1))
y = 2.0 * x[:, 0] + rng.normal(0, 0.5, size=n)

perm = rng.permutation(n)
fit_idx = perm[:1500]
cal_idx = perm[1500:2250]
te_idx = perm[2250:]

coef = np.polyfit(x[fit_idx, 0], y[fit_idx], deg=1)
score_fn = lambda X: coef[0] * X[:, 0] + coef[1]

predictor = ConformalPathwayPredictor(score_fn=score_fn, coverage=0.9)
predictor.calibrate(x[cal_idx], y[cal_idx])

intervals = predictor.predict(x[te_idx])
print(f'Empirical coverage @ target 0.9: {predictor.coverage_on(x[te_idx], y[te_idx]):.3f}')
print(f'Average interval width: {np.mean([i.width for i in intervals]):.3f}')

## 2. Bootstrap MSV intervals

Resample cells with replacement and recompute any MSV score. Percentile
intervals on the bootstrap replicates tell you how sensitive your score is
to the particular cells in the input.

In [ ]:
# Synthetic: 200 cells, each with a pathway score drawn from N(mu, sigma)
X_cells = rng.normal(loc=[0.5, -0.3, 0.1], scale=1.0, size=(200, 3))

bootstrap = BootstrapMSV(n_bootstrap=500, ci_level=0.95, seed=42)
result = bootstrap.run(X_cells, score_fn=lambda A: A.mean(axis=0))

print(result.summary())
for i, name in enumerate(['maturation', 'stress', 'drift']):
    print(f'  {name}_score = {result.point[i]:.3f} [{result.lower[i]:.3f}, {result.upper[i]:.3f}]')

## 3. Bayesian pathway GMM

Drop-in replacement for the point-estimate GMM used in the pipeline.
Returns posterior samples over component assignments so you can report
how confident a cell's subtype label is.

In [ ]:
# 3 well-separated clusters in 2D
means = np.array([[-3.0, 0.0], [3.0, 0.0], [0.0, 4.0]])
X = np.vstack([m + rng.normal(0, 0.5, size=(200, 2)) for m in means])

model = BayesianPathwayGMM(n_components=3, random_state=0).fit(X)
mode_labels = model.predict(X)
probs = model.predict_proba(X)
draws = model.sample_assignments(X, n_samples=100, random_state=1)

# Cells whose component posterior is confident vs uncertain
entropy = -np.sum(probs * np.log(probs + 1e-12), axis=1)
print(f'Mean per-cell posterior entropy: {entropy.mean():.3f}')
print(f'Posterior-sample shape (draws x cells): {draws.shape}')
print(f'Effective components (weight > 1e-3): {model.n_effective_components}')

## 4. Calibration assessment

Given predicted probabilities and observed outcomes, `CalibrationReport`
returns ECE, Brier score, and a reliability diagram — a single call
that flags miscalibration that point-accuracy metrics miss.

In [ ]:
# Well-calibrated: y ~ Bernoulli(p)
p_well = rng.uniform(0.0, 1.0, size=3000)
y_well = (rng.uniform(size=len(p_well)) < p_well).astype(int)

# Miscalibrated: y ~ Bernoulli(p^2)
p_mis = rng.uniform(0.0, 1.0, size=3000)
y_mis = (rng.uniform(size=len(p_mis)) < p_mis ** 2).astype(int)

report_well = CalibrationReport.from_predictions(y_well, p_well)
report_mis = CalibrationReport.from_predictions(y_mis, p_mis)

print('Well-calibrated:', report_well.summary())
print('Miscalibrated:  ', report_mis.summary())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
report_well.plot(ax=axes[0], show_histogram=False)
axes[0].set_title(f'Well-calibrated (ECE={report_well.ece:.3f})')
report_mis.plot(ax=axes[1], show_histogram=False)
axes[1].set_title(f'Miscalibrated (ECE={report_mis.ece:.3f})')
plt.tight_layout()
plt.show()

## Further reading

- Conformal prediction: Angelopoulos & Bates (2021). *A Gentle Introduction
  to Conformal Prediction and Distribution-Free Uncertainty Quantification.*
  arXiv:2107.07511.
- PSF v0.6 roadmap — Phase 1 Rigor Layer:
  [docs/roadmap-v06-codeberg.md](../../docs/roadmap-v06-codeberg.md)
- API reference: [docs/guides/uncertainty.md](../../docs/guides/uncertainty.md)